## Packages

In [ ]:
library(tidyverse)
library(lme4)
library(lmerTest)
library(broom)
library(broom.mixed)
library(ggeffects)
library(emmeans)
library(performance)
library(patchwork)
library(cowplot)
library(modelsummary)
library(datawizard)
library(see)
# source("scripts/simulate_lmm_data.R")

In [ ]:
simulate_lmm_data <- function() {
 
set.seed(123)
 
n_subjects <- 30
n_time <- 6
 
subject_df <- tibble(
  subject = factor(1:n_subjects),
  b0 = rnorm(n_subjects, mean = 0, sd = 6),
  b1 = rnorm(n_subjects, mean = 0, sd = 1.2)
)
 
df <- expand_grid(
  subject = factor(1:n_subjects),
  time = 0:(n_time - 1)
) |>
  left_join(subject_df, by = "subject") |>
  mutate(
    y = 20 + 2.5 * time + b0 + b1 * time + rnorm(n(), mean = 0, sd = 2)
  )
  return(df)
  }

In [ ]:
#  Organize Function (equivalent of dplyr's arrange function). From https://stackoverflow.com/questions/75667332/base-analogue-of-arrange-in-pipelines
organize <- function (df, ..., na.last = TRUE, decreasing = FALSE,
                      method = c("auto", "shell", "radix"), drop = FALSE) {
  method <- match.arg(method)
  dots <- eval(substitute(alist(...)))
  exprs <- lapply(dots, FUN = \(e) eval(e, df, parent.frame())) 
  arg_ls <- c(exprs,
              na.last = na.last,
              decreasing = decreasing,
              method = method
              )
  df[do.call("order", arg_ls), , drop = drop]
}

In [ ]:
tbl1 <- read.csv("033_ADNI Merge Cohort Final")

# TAU

## Relationship between Tau & Time

In [ ]:
# Scaling Covariates/Predictors
tbl1$VISCODEnum_z <- scale(tbl1$VISCODEnum)

In [ ]:
# Base model - scaled & refit with Bobyqa Optimizer
m2 <- lmer(
    TAU ~ VISCODEnum_z + (VISCODEnum_z | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m2)

In [ ]:
# Predicting
tbl1 <- tbl1 |>
    mutate(pred_m2 = predict(m2))

In [ ]:
# Graphing m2 w/ Random slope and intercept
m2_graph <- ggplot(tbl1, aes(VISCODEnum, TAU, colour = RID)) +
    geom_point(alpha = 0.5) +
    geom_line(aes(y = pred_m2, group = RID), linewidth = 0.9) +
    guides(colour = "none") +
    labs(
        title = "Mixed model with random intercept and slope: Tau trajectory over time",
        subtitle = "Different subject baselines and slopes",
        x = "Months since baseline",
        y = "CSF Tau"
      ) +
    theme_minimal(base_size = 13) 
m2_graph


In [ ]:
# Population predicted relationship
m2_pred <- ggpredict(m2, terms = "VISCODEnum_z") |>
  plot() +
  labs(
    title = "",
    x = "Months Since Baseline (scaled)",
    y = "CSF t-Tau"
  ) +
  theme_minimal(base_size = 13)
m2_pred

In [ ]:
# Diagnostics
cm <- performance::check_model(m2)
plot(cm, n_columns = 2)

### Adjusted For TYG

In [ ]:
# M3
# Scaling TYG
tbl1$TYG_z <- scale(tbl1$TYG)
# Refit with Bobyqa Optimizer
m3 <- lmer(
    TAU ~ VISCODEnum_z + TYG_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m3)

### Adjusted For TYG-BMI

In [ ]:
# M4
# Scaling TYG-BMI
tbl1$TYG_BMI_z <- scale(tbl1$TYG_BMI)
# Refit with Bobyqa Optimizer
m4 <- lmer(
    TAU ~ VISCODEnum_z + TYG_BMI_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m4)

### Adjusted For TG/HDL

In [ ]:
# M5
# Scaling TG/HDL
tbl1$TG_HDL_z <- scale(tbl1$TG_HDL_Ratio)
# Refit with Bobyqa Optimizer
m5 <- lmer(
    TAU ~ VISCODEnum_z + TG_HDL_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m5)

### Adjusted For METS-IR

In [ ]:
# M6
# Scaling METS-IR
tbl1$METSIR_z <- scale(tbl1$METS_IR)
# Refit with Bobyqa Optimizer
m6 <- lmer(
    TAU ~ VISCODEnum_z + METSIR_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m6)

### Comparing IR Models

In [ ]:
modelsummary(
  list(
    "No IR Covariate" = m2,
    "TYG" = m3,
    "TYG-BMI" = m4,
    "TG/HDL" = m5, 
    "METS-IR" = m6
  ),
  statistic = "({std.error})"
)

In [ ]:
# METS-IR is best aligned with TAU trajectory

## Trying Different Covariates (General)

In [ ]:
colnames(tbl1)

In [ ]:
# METS-IR & BMI
tbl1$BMI_z <- scale(tbl1$BMI)

m7 <- lmer(
    TAU ~ VISCODEnum_z + METSIR_z + BMI_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m7)

In [ ]:
# BMI Alone
m8 <- lmer(
    TAU ~ VISCODEnum_z + BMI_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m8)

In [ ]:
# APOE Alone
m9 <- lmer(
    TAU ~ VISCODEnum_z + APOE4 + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m9)

In [ ]:
# APOE & METS-IR
m10 <- lmer(
    TAU ~ VISCODEnum_z +  METSIR_z + APOE4 + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m10)

In [ ]:
# APOE & METS-IR & BMI
m11 <- lmer(
    TAU ~ VISCODEnum_z +  BMI_z + METSIR_z + APOE4 + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m11)

### Comparing Models

In [ ]:
modelsummary(
  list(
    "BMI Alone" = m8,
    "APOE Alone" = m9,
    "METS-IR & BMI" = m7,
    "METS-IR & APOE" = m10, 
    "METS-IR & BMI & APOE" = m11
  ),
  statistic = "({std.error})"
)

## Trying Different Covariates (Demographics)

In [ ]:
colnames(tbl1)

In [ ]:
# Scaling
tbl1$AGE_z <- scale(tbl1$AGE)
tbl1$PTEDUCAT_z <- scale(tbl1$PTEDUCAT)

In [ ]:
# Demog + METS-IR
m12 <- lmer(
    TAU ~ VISCODEnum_z + METSIR_z + PTGENDER + PTEDUCAT_z + PTRACCAT + AGE_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))

In [ ]:
# Demog + METS-IR + APOE
m13 <- lmer(
    TAU ~ VISCODEnum_z + METSIR_z + PTGENDER + PTEDUCAT_z + PTRACCAT + AGE_z + APOE4 + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))

In [ ]:
# Demographics Alone
m14 <- lmer(
    TAU ~ VISCODEnum_z + PTGENDER + PTEDUCAT_z + PTRACCAT + AGE_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))

In [ ]:
modelsummary(
  list(
    "Demographics Alone" = m14,
    "Demographics + METS-IR" = m12,
    "Demographics + METS-IR + APOE" = m13
  ),
  statistic = "({std.error})"
)

## Trying Different Covariates (Metabolic)

In [ ]:
colnames(tbl1)

In [ ]:
# Scaling
tbl1$GLUCOSE_z <- scale(tbl1$GLUCOSE)
tbl1$TOTAL_TG_z <- scale(tbl1$TOTAL_TG)
tbl1$HDL_C_z <- scale(tbl1$HDL_C)

In [ ]:
# Metabolic Only
m15 <- lmer(
    TAU ~ VISCODEnum_z + GLUCOSE_z + TOTAL_TG_z + HDL_C_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))

In [ ]:
# Metabolic + METS-IR
m16 <- lmer(
    TAU ~ VISCODEnum_z + METSIR_z + GLUCOSE_z + TOTAL_TG_z + HDL_C_z + (VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))

In [ ]:
# Metabolic + METS-IR + Demog + APOE
m17 <- lmer(
    TAU ~ VISCODEnum_z + METSIR_z + APOE4 + PTGENDER + PTEDUCAT_z + PTRACCAT + AGE + APOE4 + GLUCOSE_z + TOTAL_TG_z + HDL_C_z +(VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))

In [ ]:
# Metabolic + METS-IR + Demog
m18 <- lmer(
    TAU ~ VISCODEnum_z + METSIR_z + PTGENDER + PTEDUCAT_z + PTRACCAT + AGE + GLUCOSE_z + TOTAL_TG_z + HDL_C_z +(VISCODEnum | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))

In [ ]:
modelsummary(
  list(
    "Metabolic Covariates Only" = m15,
    "Metabolic + METS-IR" = m16,
    "Metabolic + METS-IR + Demographics" = m18,
    "Metabolic + METS-IR + Demographics + APOE" = m17
  ),
  statistic = "({std.error})"
)

In [ ]:
# Overall, this shows us the different imapcts of the different covariates on the dataset. 

# ABETA

## Relationship between ABETA & Time

In [ ]:
# Base model - scaled & refit with Bobyqa Optimizer
m1_a <- lmer(
    ABETA_clean ~ VISCODEnum_z + (VISCODEnum_z | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m1_a)

In [ ]:
# CSF ABETA Significantly decreases over time (suggesting progression)

In [ ]:
tbl1 <- tbl1 |>
    mutate(pred_m1_a = predict(m1_a))

In [ ]:
# Graphing m1_a w/ Random slope and intercept
m1_a_graph <- ggplot(tbl1, aes(VISCODEnum, ABETA_clean, colour = RID)) +
    geom_point(alpha = 0.5) +
    geom_line(aes(y = pred_m1_a, group = RID), linewidth = 0.9) +
    guides(colour = "none") +
    labs(
        title = "Mixed model with random intercept and slope: ABETA trajectory over time",
        subtitle = "Different subject baselines and slopes",
        x = "Months since baseline",
        y = "CSF ABETA"
      ) +
    theme_minimal(base_size = 13) 
m1_a_graph

In [ ]:
# Population predicted relationship
m1_a_pred <- ggpredict(m1_a, terms = "VISCODEnum_z") |>
  plot() +
  labs(
    title = "",
    x = "Months Since Baseline (scaled)",
    y = "CSF AB"
  ) +
  theme_minimal(base_size = 13)
m1_a_pred

In [ ]:
# Diagnostics
cm <- performance::check_model(m1_a)
plot(cm, n_columns = 2)

# PTAU

## Relationship between PTAU & Time

In [ ]:
# Base model - scaled & refit with Bobyqa Optimizer
m1_p <- lmer(
    PTAU_clean ~ VISCODEnum_z + (VISCODEnum_z | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m1_p)

In [ ]:
# PTAU signficiantly decreases over time

In [ ]:
# Predicting
tbl1 <- tbl1 |>
    mutate(pred_m1_p = predict(m1_p))

In [ ]:
# Graphing m1_p w/ Random slope and intercept
m1_p_graph <- ggplot(tbl1, aes(VISCODEnum, PTAU_clean, colour = RID)) +
    geom_point(alpha = 0.5) +
    geom_line(aes(y = pred_m1_p, group = RID), linewidth = 0.9) +
    guides(colour = "none") +
    labs(
        title = "Mixed model with random intercept and slope: PTAU trajectory over time",
        subtitle = "Different subject baselines and slopes",
        x = "Months since baseline",
        y = "CSF PTAU"
      ) +
    theme_minimal(base_size = 13) 
m1_p_graph

In [ ]:
# Population predicted relationship
m1_p_pred <- ggpredict(m1_p, terms = "VISCODEnum_z") |>
  plot() +
  labs(
    title = "",
    x = "Months Since Baseline (scaled)",
    y = "CSF p-Tau"
  ) +
  theme_minimal(base_size = 13)
m1_p_pred

In [ ]:
# Diagnostics
cm <- performance::check_model(m1_p)
plot(cm, n_columns = 2)

# Comparisons

In [ ]:
# Model Summary Comparisons
modelsummary(
  list(
    "AB" = m1_a,
    "TAU" = m2,
    "PTAU" = m1_p
  ),
  statistic = "({std.error})")

In [ ]:
cowplot::plot_grid(
    m2_graph,
    m1_a_graph,
    m1_p_graph,
    labels = c("A", "B", "C"),
      nrow = 1,
      align = "hv")

In [ ]:
plot1 <- cowplot::plot_grid(
    m1_a_pred,
    m2_pred,
    m1_p_pred,
    labels = c("A", "B", "C"),
      nrow = 1,
      align = "hv")
plot1

In [ ]:
?ggsave

In [ ]:
ggsave("038_plot1.png", plot1, width = 10, height = 5)

In [ ]:
# Grouping baseline diagnosis such that CN + SMC = CN and EMCI + LMCI = MCI
tbl1 <- tbl1 |>
    mutate(DIAGNOSIS_bl = if_else(DX_bl == "AD", "AD", 
                                  if_else(DX_bl == "EMCI", "MCI", 
                                  if_else(DX_bl == "LMCI", "MCI", 
                                  if_else(DX_bl == "SMC", "CN", "CN")))))
table(tbl3$DIAGNOSIS_bl)

In [ ]:
write.csv(tbl1, "036_ADNI Merge Modelling Cohort")

In [ ]:
colnames(tbl1)